# 04 · Hypotheses and statistics

**Why.** Put numbers on the associations seen in stage 3, with effect sizes and intervals, while being honest that these are associations and not causes.

**Method.** 25 planned two-sided tests: chi-square for the yes/no and ordered answers, Mann–Whitney U for BMI, health days and the ordered bands. Benjamini–Hochberg correction across all 25. Effect sizes: odds ratios (yes/no), Cramér's V (multi-level) and rank-biserial correlation (Mann–Whitney). Details: [statistical methods](../docs/statistics.md).

In [1]:
# Every notebook reads the committed aggregate reports, so it runs from a fresh clone without the survey data.
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENDPOINT = "combined"  # "prediabetes" shows the sensitivity analysis instead
REPORTS = ROOT / "reports"
STAGE_1_4 = REPORTS / ("uploaded-combined" if ENDPOINT == "combined" else "prediabetes")
MODELING = REPORTS / "modeling" / ENDPOINT
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 100)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def figure(path, width=860):
    display(Image(filename=str(path), width=width))

In [2]:
hypotheses = pd.read_csv(STAGE_1_4 / "hypotheses.csv")
print(f"{len(hypotheses)} tests; {int((hypotheses.p_adjusted_bh < 0.05).sum())} significant after BH correction")
columns = ["feature", "test", "effect_name", "effect_size", "ci_lower", "ci_upper", "odds_ratio_per_unit",
           "p_adjusted_bh"]
hypotheses[columns].sort_values("effect_size", ascending=False, key=abs)

25 tests; 25 significant after BH correction


,feature,test,effect_name,effect_size,ci_lower,ci_upper,odds_ratio_per_unit,p_adjusted_bh
2,CholCheck,chi_square,odds ratio (code 1 vs 0),5.868,5.231,6.583,NaN,5.329e-256
0,HighBP,chi_square,odds ratio (code 1 vs 0),4.782,4.668,4.898,NaN,0.000e+00
12,DiffWalk,chi_square,odds ratio (code 1 vs 0),3.696,3.608,3.785,NaN,0.000e+00
5,HeartDiseaseorAttack,chi_square,odds ratio (code 1 vs 0),3.511,3.411,3.614,NaN,0.000e+00
1,HighChol,chi_square,odds ratio (code 1 vs 0),3.242,3.169,3.315,NaN,0.000e+00
4,Stroke,chi_square,odds ratio (code 1 vs 0),2.968,2.846,3.096,NaN,0.000e+00
3,Smoker,chi_square,odds ratio (code 1 vs 0),1.411,1.381,1.441,NaN,2.810e-219
11,NoDocbcCost,chi_square,odds ratio (code 1 vs 0),1.408,1.360,1.459,NaN,1.009e-81
10,AnyHealthcare,chi_square,odds ratio (code 1 vs 0),1.209,1.147,1.275,NaN,1.329e-12
13,Sex,chi_square,odds ratio (code 1 vs 0),1.177,1.152,1.202,NaN,2.896e-50


## One example in full

In [3]:
hypotheses.set_index("feature").loc["HighBP", ["H0", "H1", "statistic", "p_value", "effect_size", "ci_lower", "ci_upper", "interpretation"]].to_frame("HighBP")

,HighBP
H0,Predictor and target are independent
H1,Predictor and target are associated
statistic,18539.073
p_value,0.0
effect_size,4.782
ci_lower,4.668
ci_upper,4.898
interpretation,Evidence of unadjusted association after BH correction; not causation


**Takeaway.** With 250,000 respondents almost every test is "significant", so the effect sizes and intervals matter more than the p-values (tiny p-values can underflow to 0). These are unadjusted associations in an unweighted survey extract: people with high blood pressure are far more likely to report diabetes, but that is not evidence that one causes the other, and answers such as `CholCheck` partly measure contact with doctors (who make the diagnosis). Stage 6 shows several of these associations shrink or reverse once the other answers are held fixed.

This is a research prototype for risk stratification, not a diagnostic or clinical decision tool. The outcome is self-reported, the data are cross-sectional and unweighted, and associations are not causes.